# Linear Regression
## The Least Squares method

Consider the traditional multiple linear regression problem where we have input data described by $d>1$ features $\{x_1, x_2, \dots, x_d\}$. The model is defined as:
\begin{equation}
    y = w_0 + w_1 x_1 + w_2 x_2 + \dots + w_d x_d
\end{equation}
In the following, we will use the *diabetes* dataset from the Scikit-learn library to illustrate this case:

In [7]:
from sklearn import datasets
X, y = datasets.load_diabetes(scaled=False, return_X_y=True)
print(X)

[[59.      2.     32.1    ...  4.      4.8598 87.    ]
 [48.      1.     21.6    ...  3.      3.8918 69.    ]
 [72.      2.     30.5    ...  4.      4.6728 85.    ]
 ...
 [60.      2.     24.9    ...  3.77    4.1271 95.    ]
 [36.      1.     30.     ...  4.79    5.1299 85.    ]
 [36.      1.     19.6    ...  3.      4.5951 92.    ]]


We know that the least square method leads to the following closed-form solution:
\begin{equation}
    \mathbf{w} = (\mathbf{X}^T \mathbf{X})^{-1} \mathbf{X}^T \mathbf{y}
\end{equation}
with $\mathbf{X}$ the augmented input matrix.

<span style="color:blue">Implement this formulae in python (e.g. using numpy) and give the parameters w for the diabetes dataset.</span>

In [8]:
# TODO: augment X with a column of 1
Xa = ...
# TODO: compute the vector w with the closed-form solution
w = ...
print(w[:-1]) # print the ws
print(w[-1]) # print the bias term

TypeError: 'ellipsis' object is not subscriptable

<span style="color:blue">Check your results by using the Scikit-learn implementation of Linear regression. You must obtain the same exact values.</span>

In [ ]:
from sklearn.linear_model import LinearRegression

linreg = LinearRegression()
linreg.fit(X, y)
print(linreg.coef_)
print(linreg.intercept_)

### Regularized Linear Regression
Regularized linear regression aims to prevent overfitting by adding a penalty term to the loss function. For example, with the regularized least square method, the general form of the loss function is:
\begin{equation}
    \mathcal{L}(w) = \frac{1}{2} \sum_{i=1}^{n} (y_i - \mathbf{w}^T \mathbf{x}_i)^2 + \lambda \Omega(\mathbf{w})
\end{equation}
The two most popular regularised linear regression methods are Ridge regression and Lasso regression. The Ridge regression adds a penalty term proportional to the $L_2$ norm of the weights vector:
\begin{equation}
    \Omega(\mathbf{w}) = \|\mathbf{w}\|_2^2 = \sum_{j=1}^{d} w_j^2
\end{equation}
The Lasso regression adds a penalty term proportional to the $L_1$ norm of the weights vector:
\begin{equation}
    \Omega(\mathbf{w}) = \|\mathbf{w}\|_1 = \sum_{j=1}^{d} |w_j|
\end{equation}
These methods' performances depend on the value of the hyperparameter $\lambda$. The higher the value of $\lambda$, the more the model will be regularized. For a given dataset, there are no general rules to set the value of $\lambda$. It is usually set by cross-validation. 

Scikit-learn provides ready-to-use tools for performing hyperparameter selection by cross-validation. The following is a typical example of how to use these tools: 

In [ ]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split, GridSearchCV

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3333, random_state=42)
alphas = np.logspace(-4, 4, 100)
ridge = Ridge()
grid = GridSearchCV(estimator=ridge, param_grid=dict(alpha=alphas), cv=5, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)
print(grid.best_params_)
print(grid.best_score_)

<span style="color:blue">By searching in the Scikit-learn documentation, explain the protocol used in this code to select the best hyperparameters.</span>


Comment on the code: *the `GridSearchCV` class is used to perform hyperparameter selection by cross-validation. The `param_grid` parameter is a dictionary containing the hyperparameters to be tested. Here, the candidate values for $\lambda$ are sampled logarithmically between $10^{-4}$ and $10^4$. The `cv` parameter is the number of folds in the cross-validation. The `scoring` parameter is the metric used to evaluate the model. In this case, the negative mean squared error is used.*

After having selected the best hyperparameters on the trainset, we can evaluate the model's performance on the test set:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ridge = grid.best_estimator_
ridge.fit(X_train, y_train)
predictions = ridge.predict(X_test)
print(mean_squared_error(y_test, predictions))
print(mean_absolute_error(y_test, predictions))
print(r2_score(y_test, predictions))

<span style="color:blue">Taking inspiration from the above example, propose a python script that compares the ridge regression method with the lasso regression method, each using the best possible hyperparametrisation.</span>

<span style="color:blue">What is your analysis of these results?</span>

### Feature scaling
In the previous examples, we used the raw data. However, it is often recommended to standardize the data before applying linear regression. This standardization (or Z-score normalization) can be done by subtracting the mean and dividing by the standard deviation of each feature:
\begin{equation}
    \mathbf{x}_i = \frac{\mathbf{x}_i - \bar{\mathbf{x}}}{\sigma}, \quad \forall i \in \{1, 2, \dots, n\}
\end{equation}
where $\bar{\mathbf{x}}$ is the mean of $\mathbf{x}_i \in \mathcal{D}_{train}$ and $\sigma$ is the corresponding standard deviation.

Here again, one can use Scikit-learn to perform this operation:

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_diabetes
from sklearn.linear_model import LinearRegression

X, y = load_diabetes(return_X_y=True, scaled=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3333, random_state=42)
scaler = StandardScaler()
X_train_std = scaler.fit_transform(X_train)
X_test_std = scaler.transform(X_test)
linreg = LinearRegression().fit(X_train_std, y_train)
print(linreg.score(X_test_std, y_test))


<span style="color:blue">Propose below a python script that compares the behaviors of a regular linear regression and of a Ridge regression, on raw data and on standardize data. What is your analysis of the results?</span>